In [ ]:
import pandas as pd
import numpy as np

# Sample telemetry data — use throughout this notebook
servers = pd.DataFrame({
    'server_id': ['srv-01', 'srv-02', 'srv-03', 'srv-04', 'srv-05'],
    'region':    ['us-east', 'us-east', 'us-west', 'us-west', 'us-east'],
    'tier':      ['gold', 'silver', 'gold', 'bronze', 'silver'],
})

metrics = pd.DataFrame({
    'server_id': ['srv-01', 'srv-01', 'srv-02', 'srv-02', 'srv-03',
                  'srv-03', 'srv-04', 'srv-04', 'srv-05'],
    'date':      ['2026-02-27', '2026-02-28', '2026-02-27', '2026-02-28',
                  '2026-02-27', '2026-02-28', '2026-02-27', '2026-02-28',
                  '2026-02-27'],
    'cpu':       [70, 85, 60, 95, 88, 91, 34, 37, 55],
    'mem':       [65, 72, 45, 80, 90, 92, 30, 32, 50],
})

print("servers table:")
print(servers.to_string(index=False))
print()
print("metrics table:")
print(metrics.to_string(index=False))

# Python — Pandas for Data Engineering
**Day 2 — Python Module**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>Core Insight:</strong> Pandas is the workhorse of Python data engineering.
Senior candidates know groupby internals, vectorization vs apply, and when NOT
to use pandas (use Spark instead). Run <code>p00-data</code> first to set up sample data.
</div>

## GroupBy — The Most Common Interview Topic

`groupby()` splits the DataFrame into groups, applies a function, and combines results.

**Key methods:**
- `.agg()` — reduce to one row per group
- `.transform()` — return same shape as input (window function equivalent)
- `.apply()` — most flexible, slowest
- `.filter()` — drop entire groups based on condition

In [ ]:
# GroupBy examples — all runnable on the metrics DataFrame from p00-data

# 1. Basic aggregation — one row per server
summary = metrics.groupby('server_id')['cpu'].agg(['mean', 'max', 'min']).round(2).reset_index()
print("1. Basic agg (one row per server):")
print(summary.to_string(index=False))

# 2. Multiple columns, named aggregations
by_server = metrics.groupby('server_id').agg(
    avg_cpu=('cpu', 'mean'),
    peak_cpu=('cpu', 'max'),
    avg_mem=('mem', 'mean'),
    sample_count=('cpu', 'count')
).round(2).reset_index()
print("
2. Named aggs (multiple metrics):")
print(by_server.to_string(index=False))

# 3. transform() — keeps all rows, adds group stat back
# Equivalent to SQL: AVG(cpu) OVER (PARTITION BY server_id)
metrics_with_avg = metrics.copy()
metrics_with_avg['server_avg_cpu'] = metrics.groupby('server_id')['cpu'].transform('mean').round(2)
metrics_with_avg['above_avg'] = metrics_with_avg['cpu'] > metrics_with_avg['server_avg_cpu']
print("
3. transform() — adds per-server average to every row (PARTITION BY equivalent):")
print(metrics_with_avg[['server_id','date','cpu','server_avg_cpu','above_avg']].to_string(index=False))

## Merge — JOIN Equivalent

`pd.merge()` is the standard join operation. Use it over `.join()` — it's explicit about join columns.

```python
# Inner, left, right, outer — same as SQL
result = left_df.merge(right_df, on='key_column', how='left')
```

**Key patterns:**
- Inner join: keeps only matching rows (same as SQL INNER JOIN)
- Left join: keeps all left rows, fills NaN where no right match
- Anti-join: rows in left with NO match in right (`LEFT JOIN + WHERE right.key IS NULL`)

In [ ]:
# Merge examples — joining servers + metrics

# 1. Left join (all servers, fill NaN where no metrics)
full = servers.merge(metrics, on='server_id', how='left')
print("1. Left join — all servers, NaN where no metrics:")
print(full[['server_id','tier','date','cpu']].to_string(index=False))

# 2. Anti-join: servers with NO metrics
# Pattern: left merge + filter where right key is NaN
servers_no_metrics = servers.merge(
    metrics[['server_id']].drop_duplicates(),
    on='server_id', how='left', indicator=True
).query('_merge == "left_only"').drop('_merge', axis=1)
print("
2. Anti-join — servers with NO metrics:")
print(servers_no_metrics[['server_id','tier']].to_string(index=False))

# 3. Enrichment join — add server tier to metrics
enriched = metrics.merge(servers[['server_id','tier','region']], on='server_id', how='left')
print("
3. Enriched metrics (with tier + region):")
print(enriched[['server_id','tier','region','date','cpu']].head(5).to_string(index=False))

## apply vs Vectorization — Performance Critical

**Rule:** Avoid `apply` with lambdas whenever possible. Vectorized operations are 10-100x faster on large DataFrames because they execute in NumPy's C layer, not Python.

| Method | Speed | Notes |
|--------|-------|-------|
| `apply(lambda x: ...)` | Slow | Python loop, one row at a time |
| `np.where(condition, a, b)` | Fast | C-speed, vectorized |
| `pd.cut(series, bins)` | Fast | C-speed, bin categorization |
| `series.map(dict)` | Fast | Hash lookup, no Python loop |

In [ ]:
import time

# Generate larger sample for benchmarking
large_metrics = pd.DataFrame({
    'cpu': np.random.uniform(0, 100, 100_000)
})

# Method 1: apply with lambda — Python level, slow
start = time.time()
large_metrics['cat_apply'] = large_metrics['cpu'].apply(
    lambda x: 'critical' if x > 90 else ('high' if x > 75 else 'normal')
)
apply_time = time.time() - start

# Method 2: np.where — vectorized, C speed
start = time.time()
large_metrics['cat_vectorized'] = np.where(
    large_metrics['cpu'] > 90, 'critical',
    np.where(large_metrics['cpu'] > 75, 'high', 'normal')
)
vec_time = time.time() - start

# Method 3: pd.cut — bin categorization
start = time.time()
large_metrics['cat_cut'] = pd.cut(
    large_metrics['cpu'],
    bins=[0, 75, 90, 100],
    labels=['normal', 'high', 'critical']
)
cut_time = time.time() - start

print(f"apply(lambda):  {apply_time:.4f}s")
print(f"np.where:       {vec_time:.4f}s   ({apply_time/vec_time:.1f}x faster)")
print(f"pd.cut:         {cut_time:.4f}s   ({apply_time/cut_time:.1f}x faster)")
print()

# Verify all three give the same result
print("Results agree:", (large_metrics['cat_apply'] == large_metrics['cat_vectorized']).all())
print()

# Apply IS appropriate for: complex multi-column logic that can't be vectorized
# Example: format a string using multiple columns
sample = metrics[['server_id','date','cpu']].copy()
sample['label'] = sample.apply(
    lambda row: f"{row['server_id']} on {row['date']}: {row['cpu']}%", axis=1
)
print("apply for string formatting (no vectorized equivalent):")
print(sample['label'].head(3).to_string(index=False))

## Interview Q&A

**Q: What does `groupby().transform()` do vs `groupby().agg()`?**
A: `agg()` reduces — returns one row per group. `transform()` returns a Series with the same index as the original DataFrame — like SQL window PARTITION BY. Each row gets its group's aggregate value.

**Q: When would you use pandas over Spark?**
A: Dataset fits comfortably in memory (rule of thumb: < 1-2 GB). Spark has startup overhead, cluster management complexity, and serialization cost that outweigh benefits for small data.

**Q: What's the difference between `merge()` and `join()`?**
A: `merge()` joins on column values — explicit, flexible. `join()` joins on index — simpler but can be confusing. Use `merge()` always unless you specifically want index-based joining.

**Q: `apply` vs vectorization — why does it matter at scale?**
A: `apply` is Python-level iteration — each row is a Python function call. Vectorized operations use NumPy's C/Fortran layer. On 1M rows, `apply` might take seconds; `np.where` takes milliseconds.

**Q: What is `indicator=True` in `merge()`?**
A: Adds a `_merge` column showing whether each row came from 'left_only', 'right_only', or 'both'. Useful for anti-joins and debugging join behavior.

## The Citi Angle

**Pandas was the bridge from APM data exports to analysis.** APM tools (CA APM, AppDynamics) exposed data via REST APIs and CSV exports. Before loading into databases, pandas was the transformation layer.

**Real pattern — enrichment pipeline:**
```python
# Read raw telemetry export
raw = pd.read_csv("apm_export_2026_02.csv")

# Enrich with server metadata from CMDB
cmdb = pd.read_csv("cmdb_servers.csv")
enriched = raw.merge(cmdb[["server_id","tier","owner","environment"]], on="server_id", how="left")

# Vectorized categorization (never apply)
enriched["risk_tier"] = np.where(
    enriched["cpu_pct"] > 90, "critical",
    np.where(enriched["cpu_pct"] > 75, "high", "normal")
)

# Group by tier for executive summary
summary = (enriched.groupby(["environment","risk_tier"])
    .agg(count=("server_id","count"), avg_cpu=("cpu_pct","mean"))
    .round(2).reset_index())
```

**Interview tie-in:** *"Pandas was our ETL glue layer at Citi — joining APM exports to CMDB metadata, vectorized categorization, groupby summaries. The same patterns as SQL, but in Python with better testability and version control."*